In [1]:
# web/canonical.py
import re
import pandas as pd

# Map raw strings -> (canonical_make, canonical_model)
CANON_MODEL_MAP = {
    # Tesla
    ("tesla", "3"): ("Tesla", "Model 3"),
    ("tesla", "model3"): ("Tesla", "Model 3"),
    ("tesla", "model 3"): ("Tesla", "Model 3"),
    ("tesla", "m3"): ("Tesla", "Model 3"),
    ("tesla", "y"): ("Tesla", "Model Y"),
    ("tesla", "modely"): ("Tesla", "Model Y"),
    ("tesla", "model y"): ("Tesla", "Model Y"),
    ("tesla", "x"): ("Tesla", "Model X"),
    ("tesla", "modelx"): ("Tesla", "Model X"),
    ("tesla", "model x"): ("Tesla", "Model X"),
    ("tesla", "s"): ("Tesla", "Model S"),
    ("tesla", "models"): ("Tesla", "Model S"),
    ("tesla", "model s"): ("Tesla", "Model S"),
    # Add common normalizations for other brands as needed:
    ("toyota", "rav4"): ("Toyota", "RAV4"),
    ("toyota", "rav 4"): ("Toyota", "RAV4"),
    ("chevrolet", "silverado 1500"): ("Chevrolet", "Silverado 1500"),
    ("chevrolet", "silverado1500"): ("Chevrolet", "Silverado 1500"),
}

def _norm(s: str) -> str:
    if pd.isna(s): return ""
    s = re.sub(r"[^a-z0-9]+", "", str(s).lower())
    return s

def canonicalize_make_model(make: str, model: str) -> tuple[str, str]:
    mk = (_norm(make))
    md = (_norm(model))
    if (mk, md) in CANON_MODEL_MAP:
        return CANON_MODEL_MAP[(mk, md)]
    # Generic Tesla catcher like "tesla model-3 long range"
    if mk == "tesla":
        if "model3" in md or md == "3" or md == "m3": return ("Tesla", "Model 3")
        if "modely" in md or md == "y": return ("Tesla", "Model Y")
        if "modelx" in md or md == "x": return ("Tesla", "Model X")
        if "models" in md or md == "s": return ("Tesla", "Model S")
    # Default formatting
    return (str(make).title().strip(), re.sub(r"\s+", " ", str(model)).strip().title())

TRIM_WORDS = [
    "base","lx","le","se","xle","xse","limited","platinum","sport","touring","long range",
    "performance","sr","sr5","lt","ltz","lariat","rubicon","trailhawk","premium","ultimate"
]

ENGINE_PAT = r"(?:(\d\.\dL)|(\dL)|(\d{1,2}cyl)|v\d)"

def infer_trim_engine(model_text: str) -> tuple[str | None, str | None]:
    if not isinstance(model_text, str): return (None, None)
    mtxt = model_text.lower()
    # engine
    eng = None
    m = re.search(ENGINE_PAT, mtxt)
    if m:
        eng = m.group(0).upper().replace("Cyl","CYL")
    # trim (first matching token from TRIM_WORDS)
    trim = None
    for w in TRIM_WORDS:
        if re.search(fr"\b{re.escape(w)}\b", mtxt):
            trim = w.title()
            break
    return (trim, eng)

def enrich_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "make" in df.columns and "model" in df.columns:
        canon = df[["make","model"]].apply(lambda r: canonicalize_make_model(r["make"], r["model"]), axis=1)
        df[["make","model"]] = pd.DataFrame(list(canon), index=df.index)
    # derive trim/engine if missing
    if "trim" not in df.columns:
        df["trim"] = df["model"].apply(lambda s: infer_trim_engine(s)[0])
    if "engine" not in df.columns:
        df["engine"] = df["model"].apply(lambda s: infer_trim_engine(s)[1])
    return df